# Unit 3 Assignment - Advanced RAG System

This notebook implements a full Advanced RAG pipeline for the Unit 3 assignment:

1. Document corpus setup (10+ AI/ML documents)
2. Hybrid retrieval (BM25 + SBERT + RRF)
3. Cross-encoder reranking
4. Query expansion (HyDE via Mistral)
5. End-to-end `advanced_rag(user_query)` pipeline
6. Naive vs Advanced comparison experiment

In [10]:
%pip install -q rank-bm25 sentence-transformers langchain-mistralai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import getpass

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

if not os.getenv("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Enter your Mistral API Key (or press Enter to skip LLM steps): ")

HAS_MISTRAL = bool(os.getenv("MISTRAL_API_KEY"))
print(f"Mistral enabled: {HAS_MISTRAL}")

Mistral enabled: True


## Part 1 - Document Corpus Setup

In [3]:
corpus = [
    "Transformers use self-attention to weigh token-to-token relationships and build contextual representations.",
    "Positional encodings inject order information so transformer models can reason about sequence position.",
    "Multi-head attention learns diverse interaction patterns by attending to different subspaces in parallel.",
    "Gradient descent minimizes loss by updating parameters opposite the gradient direction at each step.",
    "AdamW combines adaptive moments with decoupled weight decay, often improving generalization in deep networks.",
    "Learning rate scheduling such as cosine decay stabilizes training and improves convergence late in optimization.",
    "Batch normalization reduces internal covariate shift and can accelerate optimization of deep models.",
    "Dropout regularization randomly masks activations during training to reduce overfitting.",
    "Backpropagation applies the chain rule to propagate error signals from outputs to earlier layers.",
    "BERT is pre-trained with masked language modeling and then fine-tuned for downstream NLP tasks.",
    "Retrieval-Augmented Generation grounds responses in external documents instead of relying only on parametric memory.",
    "LoRA adapts large language models by training low-rank matrices while freezing most base parameters."
]

print(f"Total documents: {len(corpus)}")
for i, doc in enumerate(corpus):
    print(f"[{i}] {doc}")

Total documents: 12
[0] Transformers use self-attention to weigh token-to-token relationships and build contextual representations.
[1] Positional encodings inject order information so transformer models can reason about sequence position.
[2] Multi-head attention learns diverse interaction patterns by attending to different subspaces in parallel.
[3] Gradient descent minimizes loss by updating parameters opposite the gradient direction at each step.
[4] AdamW combines adaptive moments with decoupled weight decay, often improving generalization in deep networks.
[5] Learning rate scheduling such as cosine decay stabilizes training and improves convergence late in optimization.
[6] Batch normalization reduces internal covariate shift and can accelerate optimization of deep models.
[7] Dropout regularization randomly masks activations during training to reduce overfitting.
[8] Backpropagation applies the chain rule to propagate error signals from outputs to earlier layers.
[9] BERT is pr

## Part 2 - Hybrid Retrieval (BM25 + SBERT + RRF)

In [4]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60, sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.corpus = corpus
        self.k = k

        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        self.sbert = SentenceTransformer(sbert_model)
        self.doc_emb = self.sbert.encode(corpus, normalize_embeddings=True, show_progress_bar=False)

    @staticmethod
    def _ranks_from_scores(scores: np.ndarray) -> dict[int, int]:
        order = np.argsort(scores)[::-1]
        return {int(doc_id): rank + 1 for rank, doc_id in enumerate(order)}

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        query_tokens = query.lower().split()
        bm25_scores = np.array(self.bm25.get_scores(query_tokens))
        bm25_ranks = self._ranks_from_scores(bm25_scores)

        q_emb = self.sbert.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
        sbert_scores = self.doc_emb @ q_emb
        sbert_ranks = self._ranks_from_scores(sbert_scores)

        fused = []
        for doc_id in range(len(self.corpus)):
            b_rank = bm25_ranks[doc_id]
            s_rank = sbert_ranks[doc_id]
            rrf_score = 1.0 / (self.k + b_rank) + 1.0 / (self.k + s_rank)
            fused.append({
                "doc_id": doc_id,
                "rrf_score": float(rrf_score),
                "bm25_rank": int(b_rank),
                "sbert_rank": int(s_rank),
                "text": self.corpus[doc_id],
            })

        fused.sort(key=lambda x: x["rrf_score"], reverse=True)
        return fused[:top_k]

In [5]:
hybrid_retriever = HybridRetriever(corpus=corpus, k=60)
sample_hybrid = hybrid_retriever.retrieve("optimization techniques for training", top_k=5)
pd.DataFrame(sample_hybrid)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,doc_id,rrf_score,bm25_rank,sbert_rank,text
0,6,0.032018,1,4,Batch normalization reduces internal covariate...
1,5,0.031778,5,1,Learning rate scheduling such as cosine decay ...
2,7,0.031746,3,3,Dropout regularization randomly masks activati...
3,9,0.031054,2,7,BERT is pre-trained with masked language model...
4,11,0.031010,4,5,LoRA adapts large language models by training ...


## Part 3 - Cross-Encoder Re-Ranker

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    reranked = []
    for cand, score in zip(candidates, scores):
        item = dict(cand)
        item["cross_encoder_score"] = float(score)
        reranked.append(item)

    reranked.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Part 4 - Query Expansion (Option A: HyDE with Mistral)

In [12]:
hyde_llm = None
answer_llm = None

if HAS_MISTRAL:
    hyde_llm = ChatMistralAI(model="mistral-small-latest", temperature=0.0)
    answer_llm = ChatMistralAI(model="mistral-small-latest", temperature=0.2)

hyde_prompt = ChatPromptTemplate.from_template(
    "Generate a concise technical paragraph that would likely answer this student query. Use precise AI/ML terminology only. Query: {query}"
)

def hyde_expand(query: str) -> str:
    if hyde_llm is None:
        return query

    response = (hyde_prompt | hyde_llm).invoke({"query": query})
    return response.content.strip()

## Part 5 - End-to-End Pipeline

In [13]:
dense_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
dense_doc_emb = dense_encoder.encode(corpus, normalize_embeddings=True, show_progress_bar=False)

def naive_dense_top_doc(query: str) -> dict:
    q = dense_encoder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    sims = dense_doc_emb @ q
    idx = int(np.argmax(sims))
    return {"doc_id": idx, "score": float(sims[idx]), "text": corpus[idx]}

def generate_answer(user_query: str, reranked_docs: list[dict]) -> str:
    context = "\n".join([f"- {d['text']}" for d in reranked_docs])

    if answer_llm is None:
        return (
            "Mistral is not configured. Evidence-only fallback answer:\n\n"
            f"Question: {user_query}\n\n"
            f"Relevant docs:\n{context}"
        )

    prompt = ChatPromptTemplate.from_template(
        "You are a university AI tutor. Answer using only the provided context. If context is insufficient, say so clearly.\n\n"
        "Question: {question}\n\nContext:\n{context}"
    )
    response = (prompt | answer_llm).invoke({"question": user_query, "context": context})
    return response.content.strip()

def run_advanced_rag(user_query: str, retrieval_k: int = 6, rerank_k: int = 3) -> dict:
    expanded_query = hyde_expand(user_query)
    hybrid_candidates = hybrid_retriever.retrieve(expanded_query, top_k=retrieval_k)
    reranked = rerank(user_query, hybrid_candidates, top_k=rerank_k)
    answer = generate_answer(user_query, reranked)

    return {
        "user_query": user_query,
        "expanded_query": expanded_query,
        "hybrid_candidates": hybrid_candidates,
        "reranked": reranked,
        "answer": answer,
    }

def advanced_rag(user_query: str) -> str:
    """Full pipeline: Query Expansion -> Hybrid Retrieval -> Re-Ranking -> LLM Generation."""
    result = run_advanced_rag(user_query)
    return result["answer"]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
demo_query = "how do transformers encode meaning?"
demo = run_advanced_rag(demo_query)

print("User Query:", demo["user_query"])
print("\nHyDE Expanded Query:", demo["expanded_query"])
print("\nTop Re-ranked Documents:")
for i, d in enumerate(demo["reranked"], start=1):
    print(f"{i}. (score={d['cross_encoder_score']:.4f}) {d['text']}")
print("\nFinal Answer:")
print(demo["answer"])

User Query: how do transformers encode meaning?

HyDE Expanded Query: Transformers encode meaning through **self-attention mechanisms**, which dynamically weigh the relevance of each input token relative to others, enabling contextualized representations. The **multi-head attention** module computes multiple attention distributions in parallel, capturing diverse syntactic and semantic relationships. These representations are refined via **position-wise feed-forward networks** and **layer normalization**, while **residual connections** mitigate vanishing gradients. The **embedding layer** (combining token, positional, and segment embeddings) initializes semantic priors, which the model refines through **pre-training objectives** (e.g., masked language modeling) to encode distributional meaning. The final hidden states serve as **contextualized embeddings**, where semantic similarity is implicitly modeled via high-dimensional vector proximity in the latent space.

Top Re-ranked Documents

## Part 6 - Comparison Experiment

In [ ]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "why is AdamW used instead of vanilla SGD in large language model training?",
]

rows = []
for q in test_queries:
    naive = naive_dense_top_doc(q)
    adv = run_advanced_rag(q)
    adv_top_doc = adv["reranked"][0]["text"] if adv["reranked"] else "N/A"

    rows.append({
        "Query": q,
        "Naive RAG Top Doc": naive["text"],
        "Advanced RAG Top Doc": adv_top_doc,
        "Are they different?": "Yes" if naive["text"] != adv_top_doc else "No",
    })

comparison_df = pd.DataFrame(rows)
comparison_df

### Comparison Table (Paste into Final Submission)

Run the previous cell, then copy the generated markdown table output from the next cell into your submission section if needed.

In [ ]:
print(comparison_df.to_markdown(index=False))

## Notes

- This notebook uses HyDE (Option A) for query expansion.
- If `MISTRAL_API_KEY` is not set, HyDE and generation gracefully fall back to non-LLM behavior.
- Before submitting, run all cells so outputs are included in the notebook.